In [1]:
# Regrid population data to match 0.1x0.1 degrees

In [2]:
import xarray as xr

In [3]:
# === Path config ===
O3_DIR = "/glade/work/awells/air_quality/O3_obs/"
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"

In [4]:
# Open observations to use latitude and longitude values
o3 = xr.open_dataset(f"{O3_DIR}Delang_BME_OSDMA8_1990_2017.nc")["ozone"]

o3_lon = o3.longitude.values
o3_lat = o3.latitude.values

In [7]:
# Regrid each population year [2000, 2010, 2020, ..., 2100]
for year in range(2000, 2101, 10):
    print(f"Processing {year}")
    if year == 2000:
        pop = xr.open_dataset(f"{POP_DIR}baseYr_total_{year}.nc4")["Band1"]
    else:
        pop = xr.open_dataset(f"{POP_DIR}ssp2_total_{year}.nc4")["Band1"]

    pop_lon = pop.lon.values
    pop_lat = pop.lat.values

    lat_mask = xr.ufuncs.logical_and(pop_lat <= o3_lat.max(), pop_lat >= -55.75)  # -55.75 to match grid style of obs
    pop_mask = pop.sel(lat=lat_mask)

    # From GBD “Aggregation to each 0.1 × 0.1 grid cell was accomplished by summing the central 12 × 12 population cells.”
    pop_regrid = pop_mask.coarsen(lat=12, lon=12).sum()

    pop_regrid.to_netcdf(f"{POP_DIR}ssp2_total_regrid_{year}.nc")

print("Finished processing population regridding")

Processing 2000
Processing 2010
Processing 2020
Processing 2030
Processing 2040
Processing 2050
Processing 2060
Processing 2070
Processing 2080
Processing 2090
Processing 2100
Finished processing population regridding


In [2]:
# Concatenate regridding population data to one file
tot_pop = []
years = range(2000, 2101, 10)

for year in years:
    print(f"Processing {year}")
    pop = xr.open_dataarray(f"{POP_DIR}ssp2_total_regrid_{year}.nc")
    tot_pop.append(pop)

population_time_series = xr.concat(tot_pop, xr.DataArray(years, dims="year", name="year"))

population_time_series.to_netcdf(f"{POP_DIR}ssp2_total_regrid_{years[0]}-{years[-1]}.nc")

Processing 2000
Processing 2010
Processing 2020
Processing 2030
Processing 2040
Processing 2050
Processing 2060
Processing 2070
Processing 2080
Processing 2090
Processing 2100
